# Wa la Yahimk — Chapter 4 Experiments (Colab)
يشغّل هذا الدفتر كل تجارب الفصل 3 (المنهجية) ويولّد جداول وأشكال الفصل 4 تلقائياً.

**الترتيب:** شغّل الخلايا من الأعلى للأسفل.
1. تثبيت المكتبات  →  2. رفع البيانات  →  3. الإعدادات  →  4. اكتشاف الملفات
5. دوال التحميل والمعالجة  →  6. دوال التجارب  →  7. تشغيل التصنيف  →  8. التنبؤ الزمني  →  9. الجداول والأشكال  →  10. التحميل

## 1) Install

In [ ]:
!pip -q install xgboost imbalanced-learn shap openpyxl

## 2) Upload data
ارفع ملفات الـ CSV أو ملف ZIP. للملفات الكبيرة (أكثر من ~500MB) الأفضل رفعها على Google Drive واستخدام الخلية الاختيارية بعد هذه الخلية.

In [ ]:
from google.colab import files
import os
import zipfile

uploaded = files.upload()

BASE_DIR = "/content/Wa_la_Yahimk"
DATA_DIR = os.path.join(BASE_DIR, "data")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

for filename in uploaded:
    if filename.lower().endswith(".zip"):
        with zipfile.ZipFile(filename, "r") as z:
            z.extractall(DATA_DIR)
        print(f"Extracted: {filename}")

    elif filename.lower().endswith((".csv", ".parquet")):
        import shutil
        shutil.copy(filename, os.path.join(DATA_DIR, filename))
        print(f"Copied: {filename}")

print("\nData directory:", DATA_DIR)
print("Output directory:", OUTPUT_DIR)

### (اختياري) بدل الرفع: استخدام Google Drive
شغّل هذه الخلية فقط إذا كانت بياناتك على Drive. غيّر `DRIVE_FOLDER` لمسار مجلد البيانات (يحتوي مجلدات/ملفات CICIDS2017 و UNSW-NB15 و CICDDoS2019).

In [ ]:
USE_DRIVE = False
DRIVE_FOLDER = "/content/drive/MyDrive/Wa_la_Yahimk_data"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = DRIVE_FOLDER
    print("DATA_DIR ->", DATA_DIR)

# موصى به مع 5 seeds: احفظ المخرجات و cache في Drive كي يكمل التشغيل من حيث توقف إن انقطع Colab
PERSIST_OUTPUTS_TO_DRIVE = False
DRIVE_OUTPUT_FOLDER = "/content/drive/MyDrive/Wa_la_Yahimk_outputs"
if PERSIST_OUTPUTS_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = DRIVE_OUTPUT_FOLDER
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print("OUTPUT_DIR ->", OUTPUT_DIR)

## 3) Imports + Configuration
كل القرارات "المحددة مسبقاً" (predefined) في الفصل 3 موجودة هنا في مكان واحد. اذكر هذه القيم في الفصل 4 (قسم الإعداد التجريبي).

In [ ]:
import os, re, gc, glob, json, time, shutil, pickle, platform, warnings, random
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, MinMaxScaler
from sklearn.metrics import (accuracy_score, f1_score, precision_recall_fscore_support,
                             confusion_matrix, mean_absolute_error, mean_squared_error)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE, ADASYN
from sklearn.utils.class_weight import compute_sample_weight, compute_class_weight
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid", context="notebook")

try:
    DATA_DIR, OUTPUT_DIR
except NameError:
    BASE_DIR = "/content/Wa_la_Yahimk"
    DATA_DIR = os.path.join(BASE_DIR, "data")
    OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")
    os.makedirs(DATA_DIR, exist_ok=True)
    os.makedirs(OUTPUT_DIR, exist_ok=True)

FIG_DIR = os.path.join(OUTPUT_DIR, "figures")
TAB_DIR = os.path.join(OUTPUT_DIR, "tables")
CACHE_DIR = os.path.join(OUTPUT_DIR, "cache")
for d in (FIG_DIR, TAB_DIR, CACHE_DIR):
    os.makedirs(d, exist_ok=True)

SEED0 = 42
QUICK_TEST = False        # True = تشغيل سريع للتأكد أن كل شيء يعمل (نتائج غير صالحة للتقرير)

CFG = dict(
    DATASETS=["CICIDS2017", "UNSW-NB15", "CICDDoS2019", "CICDDoS2019-merged"],
    # ---- حجم البيانات (Colab لا يتحمل ملايين السجلات) ----
    MAX_ROWS=200_000,            # العدد التقريبي للسجلات المحتفظ بها لكل مجموعة بيانات
    MIN_KEEP_PER_CLASS=3_000,    # الفئات الصغيرة تُحفظ كاملة حتى هذا العدد (لحماية الفئات الأقلية)
    MIN_CLASS_TOTAL=30,          # الفئات الأقل من هذا العدد تُستبعد (تُسجَّل في جدول المعالجة)
    CHUNK=300_000,
    # ---- التقسيم ----
    TEST_SIZE=0.20,              # للبيانات التي ليس لها تقسيم جاهز
    VAL_SIZE=0.20,               # من بيانات التدريب
    SEEDS=[42, 43, 44, 45, 46],  # 5 seeds -> mean ± SD + اختبارات إحصائية (paired tests)
    VARY_SPLIT=False,            # True = يتغير تقسيم train/test الطبقي مع كل seed (لا ينطبق على UNSW-NB15 الرسمي)
    TUNE_ONCE=True,              # ضبط hyperparameters مرة واحدة (أول seed) ثم إعادة استخدامها لباقي الـ seeds
    COST_SENSITIVE=True,         # شرط جديد: cost-sensitive learning (class weights 'balanced') بدون oversampling
    MERGE_PAIRED_LABELS=True,    # يضيف CICDDoS2019-merged: دمج DrDoS_X مع X (17 -> 13 فئة)
    ALPHA=0.05,                  # مستوى الدلالة للاختبارات الإحصائية
    # ---- اختيار الخصائص ----
    CORR_THR=0.95,
    IMP_FACTOR=0.5,              # تُحذف الخاصية إذا كانت أهميتها < IMP_FACTOR × متوسط الأهمية
    MIN_FEATURES=10,
    # ---- معالجة عدم التوازن ----
    SMOTE_TARGET_CAP=20_000,     # أقصى عدد عينات لكل فئة بعد التوليد (لتحديد الزمن/الذاكرة)
    K_NEIGHBORS=5,
    # ---- الضبط والتجميع ----
    WEIGHT_GRID=[round(x, 1) for x in np.arange(0.1, 1.0, 0.1)],   # w_RF
    # ---- SHAP ----
    RUN_SHAP=True,
    SHAP_N=1000,
    # ---- التنبؤ الزمني ----
    RUN_TEMPORAL=True,
    WINDOW="1min",
    SEQ_LEN=10,
    LSTM_UNITS=64,
    DROPOUT=0.2,
    LR=1e-3,
    BATCH=32,
    EPOCHS=100,
    PATIENCE=10,
    TEMPORAL_SEEDS=[42, 43, 44, 45, 46],
    RESUME=True,                 # يستخدم النتائج المحفوظة إن انقطع الاتصال
)

if not CFG["MERGE_PAIRED_LABELS"]:
    CFG["DATASETS"] = [d for d in CFG["DATASETS"] if not d.endswith("-merged")]

RF_GRID = [
    dict(n_estimators=100, max_depth=None, min_samples_leaf=1),
    dict(n_estimators=200, max_depth=None, min_samples_leaf=1),
    dict(n_estimators=100, max_depth=20, min_samples_leaf=2),
    dict(n_estimators=200, max_depth=30, min_samples_leaf=2),
]
XGB_GRID = [
    dict(n_estimators=150, learning_rate=0.10, max_depth=6, subsample=0.8, colsample_bytree=0.8),
    dict(n_estimators=150, learning_rate=0.10, max_depth=10, subsample=0.8, colsample_bytree=0.8),
    dict(n_estimators=300, learning_rate=0.10, max_depth=6, subsample=0.8, colsample_bytree=0.8),
    dict(n_estimators=300, learning_rate=0.05, max_depth=10, subsample=0.8, colsample_bytree=0.8),
]

if QUICK_TEST:
    CFG.update(MAX_ROWS=20_000, MIN_KEEP_PER_CLASS=500, SHAP_N=200, EPOCHS=5, SEEDS=[42, 43, 44], TEMPORAL_SEEDS=[42, 43, 44], RESUME=False)
    RF_GRID = RF_GRID[:2]
    XGB_GRID = [dict(n_estimators=40, learning_rate=0.2, max_depth=5, subsample=0.8, colsample_bytree=0.8),
                dict(n_estimators=60, learning_rate=0.2, max_depth=6, subsample=0.8, colsample_bytree=0.8)]
    RF_GRID = [dict(n_estimators=40, max_depth=None, min_samples_leaf=1),
               dict(n_estimators=60, max_depth=20, min_samples_leaf=2)]

# هل يوجد GPU؟ (XGBoost يستفيد منه)
XGB_DEVICE = "cuda" if shutil.which("nvidia-smi") else "cpu"
BENIGN_NAMES = {"BENIGN", "Normal"}

ENV = dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,
           xgb_device=XGB_DEVICE, cpu_count=os.cpu_count())
import sklearn, xgboost, imblearn
ENV.update(sklearn=sklearn.__version__, xgboost=xgboost.__version__, imblearn=imblearn.__version__)
try:
    import shap
    ENV["shap"] = shap.__version__
except Exception:
    pass
print(json.dumps(ENV, indent=2))


def set_seed(s):
    random.seed(s)
    np.random.seed(s)


def savefig(name):
    path = os.path.join(FIG_DIR, name)
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.show()
    print("saved:", path)

## 4) Discover dataset files
يدعم الدفتر ملفات CSV وكذلك Parquet (مثل نسخة Kaggle المنظّفة من CIC-DDoS2019).
يتعرف الكود على كل مجموعة بيانات من اسم الملف/المجلد. إن لم يتعرف عليها، حدّد المسارات يدوياً في `MANUAL_PATHS`.

In [ ]:
MANUAL_PATHS = {
    # "CICIDS2017":  ["/content/.../Monday-WorkingHours.pcap_ISCX.csv", ...],
    # "UNSW-NB15":   ["/content/.../UNSW_NB15_training-set.csv", "/content/.../UNSW_NB15_testing-set.csv"],
    # "CICDDoS2019": ["/content/.../DrDoS_DNS.csv", ...],
}

CICDDOS_NAMES = {"syn.csv", "tftp.csv", "udplag.csv", "udp.csv", "ldap.csv", "mssql.csv",
                 "netbios.csv", "portmap.csv", "ntp.csv", "dns.csv", "snmp.csv", "ssdp.csv"}


def which_dataset(path):
    p = path.lower().replace("\\", "/")
    rel = p.replace(DATA_DIR.lower().replace("\\", "/"), "")
    base = os.path.basename(p)
    if "__macosx" in p or base.startswith("."):
        return None
    if any(k in base for k in ["nusw-nb15_features", "unsw-nb15_list", "unsw-nb15_gt", "nb15_gt"]):
        return None
    if any(k in rel for k in ["workinghours", "cicids", "machinelearningcve", "trafficlabelling", "generatedlabelledflows"]):
        return "CICIDS2017"
    if "unsw" in rel or "nb15" in rel:
        return "UNSW-NB15"
    if any(k in rel for k in ["ddos", "drdos", "01-12", "03-11"]) or base in CICDDOS_NAMES:
        return "CICDDoS2019"
    if re.search(r"-(training|testing)\.parquet$", base):      # نسخة Kaggle المنظّفة (dhoogla) من CIC-DDoS2019
        return "CICDDoS2019"
    return None


def discover_files():
    csvs = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True)
                  + glob.glob(os.path.join(DATA_DIR, "**", "*.parquet"), recursive=True))
    found = {d: [] for d in CFG["DATASETS"]}
    for f in csvs:
        d = which_dataset(f)
        if d in found:
            found[d].append(f)
    for d, lst in MANUAL_PATHS.items():
        found[d] = list(lst)
    return found


FILES = discover_files()
rows = []
for d, lst in FILES.items():
    for f in lst:
        rows.append(dict(Dataset=d, File=os.path.relpath(f, DATA_DIR) if f.startswith(DATA_DIR) else f,
                         Size_MB=round(os.path.getsize(f) / 1e6, 1)))
FILES_DF = pd.DataFrame(rows)
display(FILES_DF if len(FILES_DF) else "No CSV files found. Check DATA_DIR / upload step.")
if CFG["MERGE_PAIRED_LABELS"] and FILES.get("CICDDoS2019"):
    FILES["CICDDoS2019-merged"] = FILES["CICDDoS2019"]      # نفس الملفات، تسميات مدموجة
for d in CFG["DATASETS"]:
    if not FILES.get(d):
        print(f"[WARN] No files found for {d} — it will be skipped.")

## 5) Loading + preprocessing (Section 3.3.1 – 3.3.4)
- قراءة على دفعات (chunks) حتى لا تنفد الذاكرة.
- **العد الأولي** لكل فئة ثم **أخذ عينة طبقية** (الفئات الصغيرة تُحفظ كاملة). هذه خطوة معالجة يجب ذكرها في الفصل 4.
- كل خطوة حذف تُسجَّل (Rows before/after) لتظهر في جدول المعالجة.

In [ ]:
UNSW_RAW_COLS = ["srcip", "sport", "dstip", "dsport", "proto", "state", "dur", "sbytes", "dbytes", "sttl", "dttl",
                 "sloss", "dloss", "service", "Sload", "Dload", "Spkts", "Dpkts", "swin", "dwin", "stcpb", "dtcpb",
                 "smeansz", "dmeansz", "trans_depth", "res_bdy_len", "Sjit", "Djit", "Stime", "Ltime", "Sintpkt",
                 "Dintpkt", "tcprtt", "synack", "ackdat", "is_sm_ips_ports", "ct_state_ttl", "ct_flw_http_mthd",
                 "is_ftp_login", "ct_ftp_cmd", "ct_srv_src", "ct_srv_dst", "ct_dst_ltm", "ct_src_ltm",
                 "ct_src_dport_ltm", "ct_dst_sport_ltm", "ct_dst_src_ltm", "attack_cat", "Label"]

ID_COLS = {"flow id", "source ip", "destination ip", "src ip", "dst ip", "timestamp", "unnamed: 0", "unnamed: 0.1",
           "id", "srcip", "dstip", "stime", "ltime", "source port", "src port", "sport", "simillarhttp", "similarhttp", "__index_level_0__", "index"}


_MERGE = False                      # يُضبط في prepare_dataset للنسخة -merged
BASE = lambda d: d.replace("-merged", "")


def norm_label(x):
    x = re.sub(r"\s+", " ", str(x).encode("ascii", "ignore").decode("ascii")).strip()
    if _MERGE:                       # CICDDoS2019-merged: DrDoS_X -> X (يدمج الأزواج المتزاوجة)
        x = re.sub(r"^drdos[_\-\s]*", "", x, flags=re.I)
    k = re.sub(r"[^a-z0-9]", "", x.lower())
    if k == "benign":
        return "BENIGN"
    if k == "normal":
        return "Normal"
    if k == "udplag":
        return "UDP-lag"
    if k == "backdoors":
        return "Backdoor"
    return x


def map_labels(s):
    s = s.astype(object)
    mp = {v: norm_label(v) for v in pd.unique(s.dropna())}
    return s.map(mp)


def clean_col(c):
    """يزيل BOM (ï»¿ أو \ufeff) الذي يظهر في أول عمود لبعض ملفات CSV (مثل عمود id في UNSW-NB15)."""
    return re.sub(r"^(\u00ef\u00bb\u00bf|\ufeff)", "", str(c)).strip()


def find_label_col(cols, ds):
    low = {clean_col(c).lower(): c for c in cols}
    return low.get("attack_cat") if ds == "UNSW-NB15" else low.get("label")


def parse_ts(s, ds):
    try:
        t = pd.to_datetime(s, dayfirst=(ds == "CICIDS2017"), errors="coerce", format="mixed")
    except (TypeError, ValueError):
        t = pd.to_datetime(s, dayfirst=(ds == "CICIDS2017"), errors="coerce")
    if ds == "CICIDS2017":
        # الطوابع الزمنية في CICIDS2017 بدون AM/PM؛ الساعات 1–7 تعني PM (التجميع يبدأ ~9 صباحاً)
        t = t.where(t.dt.hour >= 8, t + pd.Timedelta(hours=12))
    return t


def read_header(f):
    if f.lower().endswith(".parquet"):
        import pyarrow.parquet as pq
        return list(pq.ParquetFile(f).schema_arrow.names)
    return list(pd.read_csv(f, nrows=0, encoding="latin1").columns)


def iter_chunks(f, usecols=None):
    """يقرأ CSV أو Parquet على دفعات."""
    if f.lower().endswith(".parquet"):
        import pyarrow.parquet as pq
        for b in pq.ParquetFile(f).iter_batches(batch_size=CFG["CHUNK"], columns=usecols):
            yield b.to_pandas()
    elif usecols:
        yield from pd.read_csv(f, usecols=usecols, chunksize=CFG["CHUNK"], encoding="latin1", dtype=str)
    else:
        yield from pd.read_csv(f, chunksize=CFG["CHUNK"], encoding="latin1")


def count_labels(files, ds):
    cnt = {}
    for f in files:
        lc = find_label_col(read_header(f), ds)
        if lc is None:
            raise ValueError(f"Label column not found in {f}")
        for ch in iter_chunks(f, usecols=[lc]):
            s = map_labels(ch[lc]).dropna()
            for k, v in s.value_counts().items():
                cnt[k] = cnt.get(k, 0) + int(v)
    return cnt


def load_dataset(ds, files):
    """يرجع: (df بعد العينة, عدد السجلات الخام لكل فئة, جدول النوافذ الزمنية أو None)"""
    cnt = count_labels(files, ds)
    N = sum(cnt.values())
    r = min(1.0, CFG["MAX_ROWS"] / N)
    keep_p = {k: min(1.0, max(min(n, CFG["MIN_KEEP_PER_CLASS"]), n * r) / n) for k, n in cnt.items()}
    print(f"  raw records: {N:,} | sampling rate r={r:.4f}")
    rng = np.random.default_rng(SEED0)
    parts, win = [], []
    for f in files:
        fl = f.lower()
        split = "train" if "training" in fl else ("test" if "testing" in fl else "all")
        for ch in iter_chunks(f):
            ch.columns = [clean_col(c) for c in ch.columns]
            ch = ch.loc[:, ~pd.Index(ch.columns).str.contains(r"\.\d+$")]
            lc = find_label_col(ch.columns, ds)
            lab = map_labels(ch[lc])
            if BASE(ds) in ("CICIDS2017", "CICDDoS2019") and "Timestamp" in ch.columns:
                ts = parse_ts(ch["Timestamp"], ds)
                ok = ts.notna() & lab.notna()
                att = (~lab[ok].isin(BENIGN_NAMES)).astype(int)
                g = pd.DataFrame({"w": ts[ok].dt.floor(CFG["WINDOW"]).values, "a": att.values}) \
                      .groupby("w")["a"].agg(["size", "sum"])
                win.append(g)
            p = lab.map(keep_p).fillna(1.0).to_numpy(dtype=float)
            keep = rng.random(len(ch)) < p
            sub = ch.loc[keep].drop(columns=[lc]).copy()      # العمود الأصلي للفئة لا يدخل كخاصية
            sub["__label__"] = lab[keep].values
            sub["__split__"] = split
            parts.append(sub)
    df = pd.concat(parts, ignore_index=True)
    del parts
    gc.collect()
    wdf = None
    if win:
        wdf = pd.concat(win).groupby(level=0).sum()
        wdf.columns = ["total", "attack"]
    return df, cnt, wdf


def load_unsw_raw_windows(files_all):
    """نوافذ UNSW-NB15 الزمنية من الملفات الخام UNSW-NB15_1..4.csv (بدون ترويسة) إن وُجدت."""
    raw = [f for f in files_all if re.search(r"unsw-nb15_[1-4]\.csv$", f.lower())]
    if not raw:
        return None
    win = []
    for f in sorted(raw):
        for ch in pd.read_csv(f, header=None, names=UNSW_RAW_COLS, usecols=["Stime", "Label"],
                              chunksize=CFG["CHUNK"], encoding="latin1", low_memory=False):
            t = pd.to_datetime(pd.to_numeric(ch["Stime"], errors="coerce"), unit="s", errors="coerce")
            a = pd.to_numeric(ch["Label"], errors="coerce")
            ok = t.notna() & a.notna()
            g = pd.DataFrame({"w": t[ok].dt.floor(CFG["WINDOW"]).values, "a": a[ok].astype(int).values}) \
                  .groupby("w")["a"].agg(["size", "sum"])
            win.append(g)
    wdf = pd.concat(win).groupby(level=0).sum()
    wdf.columns = ["total", "attack"]
    return wdf


def preprocess(df, ds, n_raw):
    log = []

    def L(stage, before, after, note=""):
        log.append(dict(Dataset=ds, Stage=stage, Rows_before=int(before), Rows_after=int(after),
                        Rows_removed=int(before - after), Note=note))

    L("Raw records (all files)", n_raw, n_raw, "label counts from first pass")
    L("Class-aware stratified sampling", n_raw, len(df),
      f"target≈{CFG['MAX_ROWS']:,}; classes kept in full up to {CFG['MIN_KEEP_PER_CLASS']:,} rows")

    # 1) أعمدة المعرّفات
    drop_cols = [c for c in df.columns if c.strip().lower() in ID_COLS or (ds == "UNSW-NB15" and c.strip().lower() == "label")]
    df = df.drop(columns=drop_cols)
    L("Remove identifier / leakage columns", len(df), len(df), f"{len(drop_cols)} cols: {', '.join(drop_cols)[:200]}")

    feat_cols = [c for c in df.columns if c not in ("__label__", "__split__")]
    is_num = lambda c: pd.api.types.is_numeric_dtype(df[c])
    cat_cols = [c for c in feat_cols if not is_num(c)] if ds == "UNSW-NB15" else []
    num_cols = [c for c in feat_cols if c not in cat_cols]
    for c in num_cols:
        if not is_num(c):
            df[c] = pd.to_numeric(df[c], errors="coerce")
    for c in cat_cols:
        df[c] = df[c].astype(str).str.strip()

    # 2) inf -> NaN
    arr = df[num_cols].to_numpy(dtype=np.float64, na_value=np.nan)
    n_inf = int(np.isinf(arr).sum())
    df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan)
    L("Infinite values converted to NaN", len(df), len(df), f"{n_inf:,} infinite cells")

    # 3) label/feature missing
    n0 = len(df)
    df = df[df["__label__"].notna()]
    L("Remove rows with missing label", n0, len(df))
    n0 = len(df)
    df = df.dropna(subset=num_cols)
    L("Remove rows with NaN/inf feature values", n0, len(df))

    # 4) duplicates
    n0 = len(df)
    df = df.drop_duplicates(subset=feat_cols + ["__label__"], keep="first")
    L("Remove exact duplicates", n0, len(df), "applied on the retained sample")

    # 5) rare classes
    vc = df["__label__"].value_counts()
    rare = vc[vc < CFG["MIN_CLASS_TOTAL"]].index.tolist()
    n0 = len(df)
    df = df[~df["__label__"].isin(rare)]
    L("Remove classes with < %d records" % CFG["MIN_CLASS_TOTAL"], n0, len(df),
      "classes: " + (", ".join(f"{r}({vc[r]})" for r in rare) if rare else "none"))
    return df.reset_index(drop=True), feat_cols, cat_cols, log


def prepare_dataset(ds, files_all):
    global _MERGE
    _MERGE = ds.endswith("-merged")
    # UNSW-NB15: التصنيف يستخدم ملفي training/testing set فقط؛ الملفات الخام للنوافذ الزمنية
    files = files_all
    if ds == "UNSW-NB15":
        # ملاحظة: في UNSW-NB15 الرسمي ملف "training-set" (82,332 سجل) أصغر من "testing-set" (175,341)؛ نحافظ على التقسيم الرسمي كما هو.
        files = sorted([f for f in files_all if re.search(r"(training|testing)-set", f.lower())],
                       key=lambda x: 0 if "training" in x.lower() else 1)
        if not files:
            raise ValueError("UNSW-NB15: UNSW_NB15_training-set.csv / testing-set.csv not found")
    elif ds == "CICDDoS2019":
        files = sorted(files_all)
    else:
        files = sorted(files_all)
    df, cnt, wdf = load_dataset(ds, files)
    if ds == "UNSW-NB15":
        wdf = load_unsw_raw_windows(files_all)
    n_raw = sum(cnt.values())
    df, feat_cols, cat_cols, log = preprocess(df, ds, n_raw)
    if _MERGE:
        log.append(dict(Dataset=ds, Stage="Merge paired labels (DrDoS_X + X)", Rows_before=len(df), Rows_after=len(df), Rows_removed=0,
                        Note=f"{df['__label__'].nunique()} classes after merging"))

    le = LabelEncoder().fit(df["__label__"])
    y = le.transform(df["__label__"])
    classes = list(le.classes_)
    predefined = (ds == "UNSW-NB15") and set(df["__split__"].unique()) >= {"train", "test"}
    if predefined:
        tr_mask = (df["__split__"] == "train").to_numpy()
        tr_idx, te_idx = np.where(tr_mask)[0], np.where(~tr_mask)[0]
        in_train = set(np.unique(y[tr_idx]))
        keep_te = np.array([c in in_train for c in y[te_idx]])
        if (~keep_te).any():
            log.append(dict(Dataset=ds, Stage="Remove test rows of classes absent from train", Rows_before=len(te_idx),
                            Rows_after=int(keep_te.sum()), Rows_removed=int((~keep_te).sum()), Note=""))
        te_idx = te_idx[keep_te]
        split_note = "predefined training/testing sets retained"
    else:
        tr_idx, te_idx = train_test_split(np.arange(len(df)), test_size=CFG["TEST_SIZE"], stratify=y, random_state=SEED0)
        split_note = f"stratified {int((1 - CFG['TEST_SIZE']) * 100)}/{int(CFG['TEST_SIZE'] * 100)} split (seed {SEED0})"

    X = df[feat_cols].copy()
    if cat_cols:
        enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        enc.fit(X.iloc[tr_idx][cat_cols])
        X[cat_cols] = enc.transform(X[cat_cols])
    X = X.astype(np.float32)
    Xtr, Xte = X.iloc[tr_idx].reset_index(drop=True), X.iloc[te_idx].reset_index(drop=True)
    ytr, yte = y[tr_idx], y[te_idx]

    log.append(dict(Dataset=ds, Stage="Train/test split", Rows_before=len(df), Rows_after=len(df), Rows_removed=0,
                    Note=f"{split_note}; train={len(ytr):,}; test={len(yte):,}; features={len(feat_cols)}; "
                         f"categorical encoded={len(cat_cols)}"))

    cd = pd.DataFrame({"Class": classes,
                       "Train": np.bincount(ytr, minlength=len(classes)),
                       "Test": np.bincount(yte, minlength=len(classes))})
    cd["Total"] = cd.Train + cd.Test
    cd["Train_%"] = (cd.Train / cd.Train.sum() * 100).round(3)
    cd.insert(0, "Dataset", ds)
    return dict(ds=ds, X=X, y=y, Xtr=Xtr, ytr=ytr, Xte=Xte, yte=yte, classes=classes, feat_cols=feat_cols,
                prep_log=pd.DataFrame(log), class_dist=cd, windows=wdf, predefined=predefined)

## 6) Experiment functions (Sections 3.3.6 – 3.3.11, 3.3.15, 3.3.20)

In [ ]:
def make_rf(p, seed):
    return RandomForestClassifier(**p, n_jobs=-1, random_state=seed)


def make_xgb(p, seed):
    return XGBClassifier(**p, tree_method="hist", device=XGB_DEVICE, n_jobs=-1, eval_metric="mlogloss",
                         random_state=seed, verbosity=0)


def fit_model(kind, params, X, y, seed, cw=False):
    """cw=True -> cost-sensitive learning: وزن أكبر للفئات الأقل (class_weight / sample_weight = 'balanced')."""
    m = make_rf(params, seed) if kind == "RF" else make_xgb(params, seed)
    t = time.time()
    if cw and kind == "RF":
        m.set_params(class_weight="balanced")
        m.fit(X, y)
    elif cw:
        m.fit(X, y, sample_weight=compute_sample_weight("balanced", y))
    else:
        m.fit(X, y)
    return m, time.time() - t


def proba_full(m, X, K):
    p = m.predict_proba(X)
    if p.shape[1] == K:
        return p
    out = np.zeros((len(X), K), dtype=np.float32)
    out[:, np.asarray(m.classes_, dtype=int)] = p
    return out


def evaluate(y_true, y_pred, K):
    labels = np.arange(K)
    pm, rm, fm, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    pw, rw, fw, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels, average="weighted", zero_division=0)
    pc, rc, fc, sc = precision_recall_fscore_support(y_true, y_pred, labels=labels, average=None, zero_division=0)
    summ = dict(Accuracy=accuracy_score(y_true, y_pred), Precision_macro=pm, Recall_macro=rm, F1_macro=fm,
                Precision_weighted=pw, Recall_weighted=rw, F1_weighted=fw)
    per = pd.DataFrame(dict(Precision=pc, Recall=rc, F1=fc, Support=sc))
    return summ, per, confusion_matrix(y_true, y_pred, labels=labels)


# ---- Two-phase feature selection (3.3.6) ----
def feature_selection(Xs, ys, seed):
    d0 = Xs.shape[1]
    nun = Xs.nunique()
    const = nun[nun <= 1].index.tolist()
    X1 = Xs.drop(columns=const)
    sub = X1.sample(min(len(X1), 100_000), random_state=seed)
    corr = sub.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    drop_corr = [c for c in upper.columns if (upper[c] > CFG["CORR_THR"]).any()]
    X1 = X1.drop(columns=drop_corr)
    d1 = X1.shape[1]
    rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=seed).fit(X1, ys)
    imp = pd.Series(rf.feature_importances_, index=X1.columns).sort_values(ascending=False)
    sel = imp[imp >= CFG["IMP_FACTOR"] * imp.mean()].index.tolist()
    if len(sel) < CFG["MIN_FEATURES"]:
        sel = imp.index[:CFG["MIN_FEATURES"]].tolist()
    info = dict(d0=d0, constant_removed=len(const), corr_removed=len(drop_corr), d1=d1, d2=len(sel),
                importance_threshold=float(CFG["IMP_FACTOR"] * imp.mean()))
    return sel, info, imp


# ---- Class balancing (3.3.7) ----
def balance(X, y, method, seed):
    if method == "Original":
        return X, y, None
    cnt = np.bincount(y)
    target = min(cnt.max(), CFG["SMOTE_TARGET_CAP"])
    strat = {c: target for c, n in enumerate(cnt) if 0 < n < target}
    if not strat:
        return X, y, None
    k = max(1, min(CFG["K_NEIGHBORS"], min(cnt[c] for c in strat) - 1))
    sampler = SMOTE(sampling_strategy=strat, k_neighbors=k, random_state=seed) if method == "SMOTE" \
        else ADASYN(sampling_strategy=strat, n_neighbors=k, random_state=seed)
    try:
        Xb, yb = sampler.fit_resample(X, y)
        return Xb.astype(np.float32), yb, None
    except Exception as e:
        print(f"    [WARN] {method} failed: {str(e)[:120]}")
        return None, None, str(e)[:200]


def tune(kind, grid, Xs, ys, Xv, yv, seed, K):
    rows = []
    for p in grid:
        m, t = fit_model(kind, p, Xs, ys, seed)
        pr = proba_full(m, Xv, K).argmax(1)
        rows.append(dict(Model=kind, Params=json.dumps(p), Val_F1_macro=f1_score(yv, pr, average="macro", zero_division=0),
                         Val_F1_weighted=f1_score(yv, pr, average="weighted", zero_division=0), Fit_time_s=t))
        del m
    tab = pd.DataFrame(rows)
    best = json.loads(tab.loc[tab.Val_F1_macro.idxmax(), "Params"])
    tab["Selected"] = tab.Params.apply(lambda s: json.loads(s) == best)
    return best, tab


def choose_weights(p_rf, p_xgb, yv):
    rows = []
    for w in CFG["WEIGHT_GRID"]:
        pr = (w * p_rf + (1 - w) * p_xgb).argmax(1)
        rows.append(dict(w_RF=w, Val_F1_macro=f1_score(yv, pr, average="macro", zero_division=0),
                         Val_F1_weighted=f1_score(yv, pr, average="weighted", zero_division=0)))
    t = pd.DataFrame(rows)
    t["dist"] = (t.w_RF - 0.5).abs()
    best = t.sort_values(["Val_F1_macro", "dist"], ascending=[False, True]).iloc[0]
    return float(best.w_RF), t.drop(columns="dist")


# ---- SHAP (3.3.11) ----
def shap_to_array(sv):
    if isinstance(sv, list):
        return np.stack(sv, axis=0)
    sv = np.asarray(sv)
    return np.transpose(sv, (2, 0, 1)) if sv.ndim == 3 else sv[None]


def run_shap(ds, model, Xte_sel, yte, classes, feats, seed):
    import shap
    K = len(classes)
    rng = np.random.default_rng(seed)
    per = max(20, CFG["SHAP_N"] // K)
    idx = np.concatenate([rng.choice(np.where(yte == c)[0], size=min(per, (yte == c).sum()), replace=False)
                          for c in range(K) if (yte == c).any()])
    Xs, ys = Xte_sel[idx], yte[idx]
    t = time.time()
    sv = shap_to_array(shap.TreeExplainer(model).shap_values(Xs))        # (K, n, f)
    print(f"    SHAP computed on {len(idx)} samples in {time.time() - t:.0f}s")
    pred = model.predict(Xs)
    glob_imp = np.abs(sv).mean(axis=(0, 1))
    g = pd.DataFrame({"Feature": feats, "Mean_abs_SHAP": glob_imp}).sort_values("Mean_abs_SHAP", ascending=False)
    g.insert(0, "Dataset", ds)
    rows = []
    for c in range(K):
        m = np.abs(sv[c]).mean(0)
        for rank, j in enumerate(np.argsort(-m)[:5], 1):
            rows.append(dict(Dataset=ds, Class=classes[c], Rank=rank, Feature=feats[j], Mean_abs_SHAP=m[j]))
    pc = pd.DataFrame(rows)

    # global plot
    top = g.head(15).iloc[::-1]
    plt.figure(figsize=(7, 5.5))
    plt.barh(top.Feature, top.Mean_abs_SHAP, color="#2a6f97")
    plt.xlabel("Mean |SHAP value| (all classes)")
    plt.title(f"{ds}: global SHAP feature importance (XGBoost)")
    savefig(f"shap_global_{ds}.png")

    # local plots: 2 most frequent attack classes + rarest attack class (correctly classified)
    att = [c for c in range(K) if classes[c] not in BENIGN_NAMES and ((pred == ys) & (ys == c)).any()]
    sup = {c: int((yte == c).sum()) for c in att}
    chosen = sorted(att, key=lambda c: -sup[c])[:2] + ([min(att, key=lambda c: sup[c])] if att else [])
    chosen = list(dict.fromkeys(chosen))
    if chosen:
        fig, axes = plt.subplots(1, len(chosen), figsize=(5.2 * len(chosen), 4.2))
        axes = np.atleast_1d(axes)
        for ax, c in zip(axes, chosen):
            i = int(np.where((pred == ys) & (ys == c))[0][0])
            v = sv[c, i]
            order = np.argsort(-np.abs(v))[:10][::-1]
            ax.barh([f"{feats[j]} = {Xs[i, j]:.3g}" for j in order], v[order],
                    color=["#c0392b" if x > 0 else "#2874a6" for x in v[order]])
            ax.set_title(f"Local explanation — class '{classes[c]}'", fontsize=10)
            ax.set_xlabel("SHAP value (red = towards class)")
        plt.suptitle(f"{ds}: local SHAP explanations (XGBoost)", y=1.02)
        savefig(f"shap_local_{ds}.png")
    return g, pc


# ---- Main per-dataset experiment (Algorithm 1) ----
CONDITIONS = ["Original", "SMOTE", "ADASYN"] + (["CostSensitive"] if CFG["COST_SENSITIVE"] else [])


def run_seed(P, seed, si, tuned=None):
    """كل التجارب لـ seed واحد. يرجع أجزاء خفيفة قابلة للحفظ (بدون نماذج)."""
    ds, classes = P["ds"], P["classes"]
    K = len(classes)
    Xtr, ytr, Xte, yte = P["Xtr"], P["ytr"], P["Xte"], P["yte"]
    if CFG["VARY_SPLIT"] and not P["predefined"]:
        tr_i, te_i = train_test_split(np.arange(len(P["y"])), test_size=CFG["TEST_SIZE"], stratify=P["y"], random_state=seed)
        Xtr, ytr = P["X"].iloc[tr_i].reset_index(drop=True), P["y"][tr_i]
        Xte, yte = P["X"].iloc[te_i].reset_index(drop=True), P["y"][te_i]
    res, per_all, hyper, wtabs, overlap, bal_dist, fails = [], [], [], [], [], [], []
    cms, shap_g, shap_c, ref, sel_imp, cw_table = {}, None, None, {}, None, None
    set_seed(seed)
    Xs, Xv, ys, yv = train_test_split(Xtr, ytr, test_size=CFG["VAL_SIZE"], stratify=ytr, random_state=seed)

    # Phase 1 + 2 feature selection on the training sub-set only
    sel, info, imp = feature_selection(Xs, ys, seed)
    info.update(Dataset=ds, Seed=seed)
    if si == 0:
        sel_imp = (sel, imp)
        cwv = compute_class_weight("balanced", classes=np.arange(K), y=ytr)
        cw_table = pd.DataFrame({"Dataset": ds, "Class": classes, "Train_records": np.bincount(ytr, minlength=K), "Class_weight": cwv})
    print(f"  features: d0={info['d0']} -> d1={info['d1']} -> d2={info['d2']}")
    Xs_s, Xv_s, Xtr_s, Xte_s = Xs[sel].values, Xv[sel].values, Xtr[sel].values, Xte[sel].values
    Xtr_all, Xte_all = Xtr.values, Xte.values

    # Hyper-parameter selection on validation data (Original distribution, selected features)
    if tuned is None:
        best_rf, t_rf = tune("RF", RF_GRID, Xs_s, ys, Xv_s, yv, seed, K)
        best_xgb, t_xgb = tune("XGBoost", XGB_GRID, Xs_s, ys, Xv_s, yv, seed, K)
        for t in (t_rf, t_xgb):
            t.insert(0, "Seed", seed)
            t.insert(0, "Dataset", ds)
            hyper.append(t)
        print("  best RF:", best_rf, "| best XGB:", best_xgb)
    else:
        best_rf, best_xgb = tuned
        print("  hyper-parameters re-used from the first seed")

    def add(model, feats, bal, y_pred, ptrain, pinfer, nfeat, extra=None):
        s_, per, cm = evaluate(yte, y_pred, K)
        cfg = f"{model} | {feats} | {bal}"
        row = dict(Dataset=ds, Seed=seed, Model=model, Features=feats, Balancing=bal, Config=cfg,
                   n_features=nfeat, **s_, Train_time_s=ptrain, Infer_time_s=pinfer,
                   Majority_baseline_acc=float(np.bincount(yte).max() / len(yte)))
        if extra:
            row.update(extra)
        res.append(row)
        p = per.copy()
        p.insert(0, "Class", classes)
        p.insert(0, "Config", cfg)
        p.insert(0, "Seed", seed)
        p.insert(0, "Dataset", ds)
        per_all.append(p)
        if si == 0:
            cms[cfg] = cm
        return s_

    # (a) All features, original distribution — baseline
    for kind, prm in (("RF", best_rf), ("XGBoost", best_xgb)):
        m, tt = fit_model(kind, prm, Xtr_all, ytr, seed)
        t0 = time.time()
        pr = proba_full(m, Xte_all, K).argmax(1)
        add(kind, "All features", "Original", pr, tt, time.time() - t0, Xtr_all.shape[1])
        del m
    gc.collect()

    # (b) Selected features x {Original, SMOTE, ADASYN, CostSensitive}
    for bal in CONDITIONS:
        print(f"  condition: Selected | {bal}")
        cw = bal == "CostSensitive"
        if cw:
            Xb_s, yb_s, Xb_f, yb_f, e1, e2 = Xs_s, ys, Xtr_s, ytr, None, None
        else:
            Xb_s, yb_s, e1 = balance(Xs_s, ys, bal, seed)
            Xb_f, yb_f, e2 = balance(Xtr_s, ytr, bal, seed)
        if e1 or e2:
            fails.append(dict(Dataset=ds, Seed=seed, Balancing=bal, Reason=e1 or e2))
            continue
        if si == 0 and not cw:
            bd = pd.DataFrame({"Class": classes, "Count": np.bincount(yb_f, minlength=K)})
            bd.insert(0, "Balancing", bal)
            bd.insert(0, "Dataset", ds)
            bal_dist.append(bd)

        # validation stage -> ensemble weights
        rf_v, _ = fit_model("RF", best_rf, Xb_s, yb_s, seed, cw)
        xg_v, _ = fit_model("XGBoost", best_xgb, Xb_s, yb_s, seed, cw)
        w, wtab = choose_weights(proba_full(rf_v, Xv_s, K), proba_full(xg_v, Xv_s, K), yv)
        wtab.insert(0, "Balancing", bal)
        wtab.insert(0, "Seed", seed)
        wtab.insert(0, "Dataset", ds)
        wtabs.append(wtab)
        del rf_v, xg_v

        # final stage: retrain on full training data, evaluate once on the untouched test set
        rf_f, t1 = fit_model("RF", best_rf, Xb_f, yb_f, seed, cw)
        xg_f, t2 = fit_model("XGBoost", best_xgb, Xb_f, yb_f, seed, cw)
        t0 = time.time(); p_rf = proba_full(rf_f, Xte_s, K); i1 = time.time() - t0
        t0 = time.time(); p_xg = proba_full(xg_f, Xte_s, K); i2 = time.time() - t0
        pr_rf, pr_xg = p_rf.argmax(1), p_xg.argmax(1)
        pr_en = (w * p_rf + (1 - w) * p_xg).argmax(1)
        add("RF", "Selected", bal, pr_rf, t1, i1, len(sel))
        add("XGBoost", "Selected", bal, pr_xg, t2, i2, len(sel))
        add("Ensemble (RF+XGB)", "Selected", bal, pr_en, t1 + t2, i1 + i2, len(sel), extra=dict(w_RF=w, w_XGB=1 - w))

        c_rf, c_xg, c_en = pr_rf == yte, pr_xg == yte, pr_en == yte
        overlap.append(dict(Dataset=ds, Seed=seed, Balancing=bal, N_test=len(yte),
                            Both_correct=int((c_rf & c_xg).sum()), Only_RF_correct=int((c_rf & ~c_xg).sum()),
                            Only_XGB_correct=int((~c_rf & c_xg).sum()), Both_wrong=int((~c_rf & ~c_xg).sum()),
                            Ensemble_correct_when_models_disagree=int((c_en & (c_rf ^ c_xg)).sum()),
                            Ensemble_correct_when_both_wrong=int((c_en & ~c_rf & ~c_xg).sum())))
        if si == 0 and bal == "Original":
            ref = dict(xgb=xg_f, Xte_sel=Xte_s)
        else:
            del xg_f
        del rf_f, Xb_s, Xb_f
        gc.collect()

    if CFG["RUN_SHAP"] and ref:
        try:
            shap_g, shap_c = run_shap(ds, ref["xgb"], ref["Xte_sel"], yte, classes, sel_imp[0], seed)
        except Exception as e:
            print("  [WARN] SHAP failed:", str(e)[:200])
    return dict(res=pd.DataFrame(res), per=pd.concat(per_all, ignore_index=True), fs=info, sel_imp=sel_imp,
                hyper=hyper, best=(best_rf, best_xgb), weights=wtabs, overlap=overlap, bal_dist=bal_dist,
                fails=fails, cms=cms, shap=(shap_g, shap_c), cw_table=cw_table)


def run_dataset(P, seeds):
    ds, classes = P["ds"], P["classes"]
    parts, tuned = [], None
    for si, seed in enumerate(seeds):
        pf = os.path.join(CACHE_DIR, f"seed_{ds}_{seed}_of{len(seeds)}{'_quick' if QUICK_TEST else ''}.pkl")
        if CFG["RESUME"] and os.path.exists(pf):
            with open(pf, "rb") as fh:
                part = pickle.load(fh)
            print(f"  [{ds}] seed {seed} loaded from cache")
        else:
            print(f"\n=== {ds} | seed {seed} ({si + 1}/{len(seeds)}) ===")
            t0 = time.time()
            part = run_seed(P, seed, si, tuned if CFG["TUNE_ONCE"] else None)
            print(f"  seed {seed} finished in {(time.time() - t0) / 60:.1f} min")
            with open(pf, "wb") as fh:
                pickle.dump(part, fh)
        if si == 0:
            tuned = part["best"]
        parts.append(part)

    res = pd.concat([p["res"] for p in parts], ignore_index=True)
    if len(res) and res.Accuracy.max() < res.Majority_baseline_acc.iloc[0] + 0.05:
        print(f"  [WARNING] {ds}: best accuracy ({res.Accuracy.max():.3f}) is within 0.05 of the majority-class baseline "
              f"({res.Majority_baseline_acc.iloc[0]:.3f}). Check for data problems (identifier columns, split, encoding).")
    sel, imp = parts[0]["sel_imp"]
    cat = lambda key: [x for p in parts for x in p[key]]
    out = dict(res=res, perclass=pd.concat([p["per"] for p in parts], ignore_index=True),
               fs=pd.DataFrame([p["fs"] for p in parts]),
               fs_features=pd.DataFrame({"Dataset": ds, "Feature": imp.index, "RF_importance": imp.values,
                                         "Selected": [f in set(sel) for f in imp.index]}),
               hyper=pd.concat(cat("hyper"), ignore_index=True), weights=pd.concat(cat("weights"), ignore_index=True),
               overlap=pd.DataFrame(cat("overlap")),
               bal_dist=pd.concat(cat("bal_dist"), ignore_index=True) if cat("bal_dist") else pd.DataFrame(),
               fails=pd.DataFrame(cat("fails")), cms=parts[0]["cms"], classes=classes,
               shap_global=parts[0]["shap"][0], shap_class=parts[0]["shap"][1], cw_table=parts[0]["cw_table"])
    return out

## 7) Run the classification experiments (RQ1 – RQ5)
**الوقت:** 5 seeds × 4 مجموعات (CICIDS2017 / UNSW-NB15 / CICDDoS2019 / CICDDoS2019-merged) × 5 شروط تدريب قد يستغرق عدة ساعات. لتقليله: فعّل GPU (يسرّع XGBoost)، واترك `TUNE_ONCE=True`.
النتائج تُحفظ بعد **كل seed** (`RESUME=True`)، فإن انقطع الاتصال أعد التشغيل وسيكمل من حيث توقف. لا تحذف مجلد cache.
**نصيحة:** جرّب أولاً `QUICK_TEST=True` في خلية الإعدادات للتأكد أن كل شيء يعمل، ثم أعد التشغيل بـ `False`.

In [ ]:
RESULTS, PREP = {}, {}
T_START = time.time()

for ds in CFG["DATASETS"]:
    if not FILES.get(ds):
        continue
    cache = os.path.join(CACHE_DIR, f"results_{ds}_{len(CFG['SEEDS'])}seeds{'_quick' if QUICK_TEST else ''}.pkl")
    if CFG["RESUME"] and os.path.exists(cache):
        with open(cache, "rb") as fh:
            RESULTS[ds], PREP[ds] = pickle.load(fh)
        print(f"[{ds}] loaded from cache")
        continue
    print("\n" + "#" * 70 + f"\n# {ds}\n" + "#" * 70)
    try:
        P = prepare_dataset(ds, FILES[ds])
    except Exception as e:
        print(f"[ERROR] {ds} preparation failed: {e}")
        continue
    display(P["prep_log"][["Stage", "Rows_before", "Rows_after", "Rows_removed", "Note"]])
    display(P["class_dist"])
    out = run_dataset(P, CFG["SEEDS"])
    RESULTS[ds] = out
    PREP[ds] = dict(prep_log=P["prep_log"], class_dist=P["class_dist"], windows=P["windows"],
                    n_train=len(P["ytr"]), n_test=len(P["yte"]), n_features=len(P["feat_cols"]),
                    predefined=P["predefined"])
    with open(cache, "wb") as fh:
        pickle.dump((out, PREP[ds]), fh)
    del P
    gc.collect()

print(f"\nTotal time: {(time.time() - T_START) / 60:.1f} min")

## 8) Temporal prediction: LSTM vs Bi-LSTM (RQ6, Sections 3.3.12 – 3.3.16)
يحتاج أعمدة `Timestamp` (CICIDS2017 نسخة TrafficLabelling، و CICDDoS2019) أو الملفات الخام `UNSW-NB15_1..4.csv`. أي مجموعة بدونها تُتخطى مع رسالة.

In [ ]:
TEMPORAL = {}


def run_temporal(ds, win, seeds):
    import tensorflow as tf
    from tensorflow.keras import layers, callbacks, Model

    win = win[win["total"] > 0]                      # فقط النوافذ التي فيها حركة مسجّلة
    series = win["attack"].to_numpy(dtype=np.float32)
    T, w = len(series), CFG["SEQ_LEN"]
    print(f"  {ds}: {T} active windows ({CFG['WINDOW']}), attack count range [{series.min():.0f}, {series.max():.0f}]")
    if T < w * 8:
        print(f"  [SKIP] too few windows (< {w * 8}) for chronological train/val/test.")
        return None
    n_tr, n_va = int(T * 0.70), int(T * 0.15)
    scaler = MinMaxScaler().fit(series[:n_tr].reshape(-1, 1))             # fit on training period only
    sc = scaler.transform(series.reshape(-1, 1)).ravel().astype(np.float32)
    X = np.stack([sc[i:i + w] for i in range(T - w)])[..., None]
    y = sc[w:]
    tgt = np.arange(w, T)
    tr, va, te = tgt < n_tr, (tgt >= n_tr) & (tgt < n_tr + n_va), tgt >= n_tr + n_va
    y_te_orig = series[tgt[te]]
    naive = series[tgt[te] - 1]                                            # persistence reference

    def build(kind):
        inp = layers.Input((w, 1))
        h = layers.Bidirectional(layers.LSTM(CFG["LSTM_UNITS"]))(inp) if kind == "Bi-LSTM" else layers.LSTM(CFG["LSTM_UNITS"])(inp)
        h = layers.Dropout(CFG["DROPOUT"])(h)
        m = Model(inp, layers.Dense(1)(h))
        m.compile(optimizer=tf.keras.optimizers.Adam(CFG["LR"]), loss="mse")
        return m

    rows, preds, hist = [], {}, {}
    for kind in ("LSTM", "Bi-LSTM"):
        for si, seed in enumerate(seeds):
            tf.keras.utils.set_random_seed(seed)
            m = build(kind)
            es = callbacks.EarlyStopping(monitor="val_loss", patience=CFG["PATIENCE"], restore_best_weights=True)
            t0 = time.time()
            h = m.fit(X[tr], y[tr], validation_data=(X[va], y[va]), epochs=CFG["EPOCHS"], batch_size=CFG["BATCH"],
                      callbacks=[es], verbose=0)
            pr = scaler.inverse_transform(m.predict(X[te], verbose=0)).ravel()
            rows.append(dict(Dataset=ds, Model=kind, Seed=seed, MAE=mean_absolute_error(y_te_orig, pr),
                             RMSE=float(np.sqrt(mean_squared_error(y_te_orig, pr))), Epochs_run=len(h.history["loss"]),
                             Train_time_s=time.time() - t0, Params=m.count_params()))
            if si == 0:
                preds[kind], hist[kind] = pr, h.history
            tf.keras.backend.clear_session()
    rows.append(dict(Dataset=ds, Model="Persistence (reference)", Seed=0, MAE=mean_absolute_error(y_te_orig, naive),
                     RMSE=float(np.sqrt(mean_squared_error(y_te_orig, naive))), Epochs_run=0, Train_time_s=0, Params=0))
    info = dict(Dataset=ds, Windows=T, Window_size=CFG["WINDOW"], Seq_len=w, Train_windows=int(tr.sum()),
                Val_windows=int(va.sum()), Test_windows=int(te.sum()), Mean_attack_per_window=float(series.mean()),
                Max_attack_per_window=float(series.max()), Start=str(win.index.min()), End=str(win.index.max()))
    return dict(metrics=pd.DataFrame(rows), info=info, y_true=y_te_orig, preds=preds, naive=naive, hist=hist)


TF_OK = False
if CFG["RUN_TEMPORAL"]:
    try:
        import tensorflow as tf
        TF_OK = True
    except ImportError:
        print("TensorFlow not available -> temporal component skipped.")

if TF_OK:
    for ds in CFG["DATASETS"]:
        if ds not in PREP or ds.endswith("-merged"):
            continue
        win = PREP[ds]["windows"]
        if win is None or len(win) == 0:
            print(f"[{ds}] no timestamp information found -> temporal component skipped.")
            continue
        r = run_temporal(ds, win, CFG["TEMPORAL_SEEDS"])
        if r is not None:
            TEMPORAL[ds] = r
    for ds, r in TEMPORAL.items():
        display(r["metrics"])

## 9) Build Chapter-4 tables and figures
كل جدول يُحفظ في `tables/` وكذلك في ملف Excel واحد. تذكّر قاعدة الدليل: **كل نتيجة تُعرض مرة واحدة فقط (جدول أو شكل وليس الاثنين)**.

In [ ]:
def fmt_ms(df, group, metrics):
    """mean (± SD إن وُجدت عدة seeds)"""
    g = df.groupby(group, sort=False)[metrics]
    m, s, n = g.mean(), g.std(ddof=1), g.size()
    out = m.copy()
    if CFG["SEEDS"] and len(CFG["SEEDS"]) > 1:
        for c in metrics:
            out[c] = [f"{a:.4f} ± {b:.4f}" for a, b in zip(m[c], s[c].fillna(0))]
    return out.reset_index()


ALL = lambda k: pd.concat([RESULTS[d][k] for d in RESULTS], ignore_index=True)
TABLES = {}

# 4.2.1 data & preprocessing
TABLES["T01_preprocessing_log"] = pd.concat([PREP[d]["prep_log"] for d in PREP], ignore_index=True)
TABLES["T02_class_distribution"] = pd.concat([PREP[d]["class_dist"] for d in PREP], ignore_index=True)
TABLES["T03_dataset_summary"] = pd.DataFrame([dict(Dataset=d, Train=PREP[d]["n_train"], Test=PREP[d]["n_test"],
                                                   Features_after_preprocessing=PREP[d]["n_features"],
                                                   Classes=len(RESULTS[d]["classes"]),
                                                   Split="predefined" if PREP[d]["predefined"] else "stratified 80/20")
                                              for d in PREP])
# 4.2.2 feature selection
fs = ALL("fs")
fs_mean = fs.groupby("Dataset", sort=False)[["d0", "constant_removed", "corr_removed", "d1", "d2"]].mean().round(1).reset_index()
TABLES["T04_feature_selection"] = fs_mean
TABLES["T04b_feature_selection_sd"] = fs.groupby("Dataset", sort=False)[["d1", "d2"]].agg(["mean", "std"]).round(2).reset_index()
TABLES["T04b_feature_selection_sd"].columns = ["_".join([c for c in col if c]) for col in TABLES["T04b_feature_selection_sd"].columns]
TABLES["T05_selected_features"] = ALL("fs_features").query("Selected")[["Dataset", "Feature", "RF_importance"]]
TABLES["T06_hyperparameter_search"] = ALL("hyper")

# main results
res = ALL("res")
METRICS = ["Accuracy", "Precision_macro", "Recall_macro", "F1_macro", "Precision_weighted", "Recall_weighted", "F1_weighted"]
main = fmt_ms(res, ["Dataset", "Config", "Model", "Features", "Balancing"], METRICS + ["n_features", "Train_time_s", "Infer_time_s"])
TABLES["T07_all_configurations"] = main

num = res.groupby(["Dataset", "Model", "Features", "Balancing"], sort=False)[METRICS + ["Train_time_s", "Infer_time_s"]].mean().reset_index()

# RQ1: individual models vs ensemble (selected, original)
TABLES["T08_RQ1_models"] = num.query("Features=='Selected' and Balancing=='Original'")[
    ["Dataset", "Model"] + METRICS]
# RQ2: effect of feature selection
a = num.query("Features=='All features' and Balancing=='Original'").set_index(["Dataset", "Model"])
b = num.query("Features=='Selected' and Balancing=='Original' and Model!='Ensemble (RF+XGB)'").set_index(["Dataset", "Model"])
rq2 = pd.DataFrame({"F1_macro_all": a.F1_macro, "F1_macro_selected": b.F1_macro,
                    "F1_weighted_all": a.F1_weighted, "F1_weighted_selected": b.F1_weighted,
                    "Train_time_all_s": a.Train_time_s, "Train_time_selected_s": b.Train_time_s}).dropna().reset_index()
rq2["Delta_F1_macro"] = rq2.F1_macro_selected - rq2.F1_macro_all
rq2["Delta_F1_weighted"] = rq2.F1_weighted_selected - rq2.F1_weighted_all
rq2 = rq2.merge(fs_mean[["Dataset", "d0", "d2"]], on="Dataset", how="left")
TABLES["T09_RQ2_feature_selection_effect"] = rq2

# RQ3: balancing — overall + minority classes
cd = TABLES["T02_class_distribution"]
minority = {}
for d in cd.Dataset.unique():
    s = cd[cd.Dataset == d]
    m = s[s["Train_%"] < 1.0].Class.tolist() or s.nsmallest(3, "Train").Class.tolist()
    minority[d] = m
pc = ALL("perclass")
pc["Minority"] = [c in minority[d] for d, c in zip(pc.Dataset, pc.Class)]
mino = pc[pc.Minority].groupby(["Dataset", "Config"], sort=False)[["Recall", "F1"]].mean().rename(
    columns={"Recall": "Minority_mean_Recall", "F1": "Minority_mean_F1"}).reset_index()
rq3 = res.groupby(["Dataset", "Config", "Model", "Features", "Balancing"], sort=False)[["F1_macro", "F1_weighted", "Recall_macro"]].mean().reset_index()
rq3 = rq3.query("Features=='Selected'").merge(mino, on=["Dataset", "Config"], how="left")
TABLES["T10_RQ3_class_balancing"] = rq3.drop(columns="Features")
TABLES["T11_minority_classes_definition"] = pd.DataFrame([dict(Dataset=d, Minority_classes=", ".join(v)) for d, v in minority.items()])
if len(ALL("bal_dist")):
    TABLES["T12_training_distribution_after_balancing"] = ALL("bal_dist").pivot_table(
        index=["Dataset", "Class"], columns="Balancing", values="Count", aggfunc="first").reset_index()

# RQ4: ensemble
ens = res[res.Model == "Ensemble (RF+XGB)"].groupby(["Dataset", "Balancing"], sort=False)[["w_RF", "w_XGB"]].mean().reset_index()
TABLES["T13_RQ4_ensemble_weights"] = ens
ov = ALL("overlap").groupby(["Dataset", "Balancing"], sort=False).sum(numeric_only=True).drop(columns="Seed").reset_index()
TABLES["T14_RQ4_error_overlap"] = ov
TABLES["T15_validation_weight_search"] = ALL("weights")

fl = ALL("fails")
if len(fl):
    TABLES["T24_balancing_failures"] = fl       # اذكرها في الفصل 4: حالات لم ينجح فيها SMOTE/ADASYN
# per-class
TABLES["T16_per_class_metrics"] = pc.groupby(["Dataset", "Config", "Class"], sort=False)[["Precision", "Recall", "F1", "Support"]].mean().reset_index()

# targets
tg = []
for d in RESULTS:
    s = num[(num.Dataset == d) & (num.Features == "Selected") & (num.Balancing == "Original")]
    for _, r in s.iterrows():
        tg.append(dict(Dataset=d, Model=r.Model, Weighted_F1=r.F1_weighted, Macro_F1=r.F1_macro,
                       Target_weighted_F1_95_on_CICIDS2017=("Yes" if r.F1_weighted >= 0.95 else "No") if d == "CICIDS2017" else "n/a",
                       Target_macro_F1_85=("Yes" if r.F1_macro >= 0.85 else "No"),
                       Seeds_with_macro_F1_ge_85=(lambda m_: f"{int((m_ & (res.F1_macro >= 0.85)).sum())}/{int(m_.sum())}")(
                           (res.Dataset == d) & (res.Model == r.Model) & (res.Features == 'Selected') & (res.Balancing == 'Original'))))
TABLES["T17_performance_targets"] = pd.DataFrame(tg)

# cross-dataset (primary configuration = ensemble, selected, original)
TABLES["T18_cross_dataset_primary"] = num.query("Model=='Ensemble (RF+XGB)' and Features=='Selected' and Balancing=='Original'")[
    ["Dataset"] + METRICS + ["Train_time_s", "Infer_time_s"]]
best_idx = num.groupby("Dataset").F1_macro.idxmax()
TABLES["T19_best_configuration_per_dataset"] = num.loc[best_idx][["Dataset", "Model", "Features", "Balancing", "F1_macro", "F1_weighted"]]

# SHAP
sg = [RESULTS[d]["shap_global"] for d in RESULTS if RESULTS[d]["shap_global"] is not None]
if sg:
    top10 = pd.concat([g.head(10).assign(Rank=range(1, 11)) for g in sg], ignore_index=True)
    TABLES["T20_SHAP_global_top10"] = top10
    TABLES["T21_SHAP_top5_per_class"] = pd.concat([RESULTS[d]["shap_class"] for d in RESULTS if RESULTS[d]["shap_class"] is not None], ignore_index=True)

# temporal
if TEMPORAL:
    tm = pd.concat([r["metrics"] for r in TEMPORAL.values()], ignore_index=True)
    TABLES["T22_temporal_setup"] = pd.DataFrame([r["info"] for r in TEMPORAL.values()])
    TABLES["T23_temporal_results"] = tm.groupby(["Dataset", "Model"], sort=False)[["MAE", "RMSE", "Epochs_run", "Train_time_s", "Params"]].agg(["mean", "std"]).round(4).reset_index()
    TABLES["T23_temporal_results"].columns = ["_".join([c for c in col if c]) for col in TABLES["T23_temporal_results"].columns]


# cost-sensitive class weights
cwt = [RESULTS[d]["cw_table"] for d in RESULTS if RESULTS[d].get("cw_table") is not None]
if cwt:
    TABLES["T25_cost_sensitive_class_weights"] = pd.concat(cwt, ignore_index=True)

# ---------- mean ± SD + statistical tests over seeds ----------
mrec = pc[pc.Minority].groupby(["Dataset", "Seed", "Config"])["Recall"].mean().rename("Minority_mean_Recall").reset_index()
mf1 = pc[pc.Minority].groupby(["Dataset", "Seed", "Config"])["F1"].mean().rename("Minority_mean_F1").reset_index()
res2 = res.merge(mrec, on=["Dataset", "Seed", "Config"], how="left").merge(mf1, on=["Dataset", "Seed", "Config"], how="left")
SUMM = ["Accuracy", "F1_macro", "F1_weighted", "Minority_mean_Recall", "Minority_mean_F1", "Train_time_s", "Infer_time_s"]
rows_ms = []
for k, d_ in res2.groupby(["Dataset", "Model", "Features", "Balancing"], sort=False):
    r_ = dict(zip(["Dataset", "Model", "Features", "Balancing"], k))
    r_["n_seeds"] = int(d_.Seed.nunique())
    for c in SUMM:
        m_, s_ = d_[c].mean(), (d_[c].std(ddof=1) if len(d_) > 1 else np.nan)
        r_[c + "_mean"], r_[c + "_sd"] = m_, s_
        r_[c] = f"{m_:.4f} ± {s_:.4f}" if len(d_) > 1 else f"{m_:.4f}"
    rows_ms.append(r_)
TABLES["T26_main_results_mean_sd"] = pd.DataFrame(rows_ms)


def paired(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    d = a - b
    n = len(d)
    o = dict(n_pairs=n, mean_A=a.mean(), mean_B=b.mean(), mean_diff=d.mean(), sd_diff=d.std(ddof=1) if n > 1 else np.nan)
    if n >= 3 and d.std(ddof=1) > 0:
        t_, p_ = stats.ttest_rel(a, b)
        h_ = stats.t.ppf(0.975, n - 1) * d.std(ddof=1) / np.sqrt(n)
        o.update(t=t_, p_ttest=p_, ci95_low=d.mean() - h_, ci95_high=d.mean() + h_, cohen_dz=d.mean() / d.std(ddof=1))
    else:
        o.update(t=np.nan, p_ttest=(1.0 if n >= 3 and np.allclose(d, 0) else np.nan), ci95_low=np.nan, ci95_high=np.nan, cohen_dz=np.nan)
    try:
        o["p_wilcoxon"] = stats.wilcoxon(a, b).pvalue if np.any(d != 0) else 1.0
    except Exception:
        o["p_wilcoxon"] = np.nan
    return o


def holm(p):
    p = np.asarray(p, float)
    order = np.argsort(p)
    m_ = len(p)
    adj, run = np.empty(m_), 0.0
    for rank, idx in enumerate(order):
        run = max(run, (m_ - rank) * p[idx])
        adj[idx] = min(1.0, run)
    return adj


ENS = "Ensemble (RF+XGB)"
tests = []


def TST(d_, fam, metric, A, B):
    sa = res2[(res2.Dataset == d_) & (res2.Model == A[0]) & (res2.Features == A[1]) & (res2.Balancing == A[2])].set_index("Seed")[metric]
    sb = res2[(res2.Dataset == d_) & (res2.Model == B[0]) & (res2.Features == B[1]) & (res2.Balancing == B[2])].set_index("Seed")[metric]
    common = sa.index.intersection(sb.index)
    if len(common) < 2:
        return
    tests.append(dict(Dataset=d_, Family=fam, Metric=metric, A=" | ".join(A), B=" | ".join(B), **paired(sa.loc[common].values, sb.loc[common].values)))


if res.Seed.nunique() >= 2:
    for d in RESULTS:
        for m in ["F1_macro", "F1_weighted", "Accuracy"]:
            TST(d, "RQ1 models", m, ("XGBoost", "Selected", "Original"), ("RF", "Selected", "Original"))
            TST(d, "RQ1 models", m, (ENS, "Selected", "Original"), ("XGBoost", "Selected", "Original"))
            TST(d, "RQ1 models", m, (ENS, "Selected", "Original"), ("RF", "Selected", "Original"))
        for mod in ["RF", "XGBoost"]:
            for m in ["F1_macro", "F1_weighted", "Train_time_s"]:
                TST(d, "RQ2 feature selection", m, (mod, "Selected", "Original"), (mod, "All features", "Original"))
        for mod in ["RF", "XGBoost", ENS]:
            for cond in ["SMOTE", "ADASYN", "CostSensitive"]:
                for m in ["F1_macro", "F1_weighted", "Minority_mean_Recall"]:
                    TST(d, "RQ3 vs Original", m, (mod, "Selected", cond), (mod, "Selected", "Original"))
            TST(d, "RQ3 SMOTE vs ADASYN", "F1_macro", (mod, "Selected", "SMOTE"), (mod, "Selected", "ADASYN"))
            TST(d, "RQ3 CostSensitive vs SMOTE", "F1_macro", (mod, "Selected", "CostSensitive"), (mod, "Selected", "SMOTE"))
    if tests:
        tt = pd.DataFrame(tests)
        tt["p_holm"] = tt.groupby(["Dataset", "Family", "Metric"])["p_ttest"].transform(lambda s_: holm(s_.fillna(1.0)))
        tt.loc[tt.p_ttest.isna(), "p_holm"] = np.nan
        tt["significant"] = tt.p_holm < CFG["ALPHA"]
        TABLES["T27_statistical_tests"] = tt
        print("Statistical tests:", len(tt), "comparisons;", int(tt.significant.sum()), f"significant (Holm-adjusted p < {CFG['ALPHA']})")

if TEMPORAL and CFG["TEMPORAL_SEEDS"] and len(CFG["TEMPORAL_SEEDS"]) >= 3:
    ttp = []
    for d, r in TEMPORAL.items():
        mm = r["metrics"]
        l_ = mm[mm.Model == "LSTM"].set_index("Seed")
        b_ = mm[mm.Model == "Bi-LSTM"].set_index("Seed")
        pers_row = mm[mm.Model.str.startswith("Persistence")]
        common = l_.index.intersection(b_.index)
        for met in ["MAE", "RMSE"]:
            ttp.append(dict(Dataset=d, Comparison="LSTM vs Bi-LSTM (paired t-test)", Metric=met, **paired(l_.loc[common, met].values, b_.loc[common, met].values)))
            if len(pers_row):
                pers = float(pers_row[met].iloc[0])
                for nm, s_ in (("LSTM", l_[met]), ("Bi-LSTM", b_[met])):
                    t1 = stats.ttest_1samp(s_.values, pers)
                    ttp.append(dict(Dataset=d, Comparison=f"{nm} vs persistence (one-sample t-test)", Metric=met, n_pairs=len(s_),
                                    mean_A=s_.mean(), mean_B=pers, mean_diff=s_.mean() - pers, sd_diff=s_.std(ddof=1),
                                    t=t1.statistic, p_ttest=t1.pvalue))
    TABLES["T28_temporal_statistical_tests"] = pd.DataFrame(ttp)

TABLES["T29_notes_on_statistics"] = pd.DataFrame({"Note": [
    f"Seeds used: {sorted(res.Seed.unique().tolist())}; tests are paired across seeds (n = {res.Seed.nunique()}).",
    "With VARY_SPLIT=False the test set is fixed, so seeds capture training randomness only (model, validation split, oversampling), not test-set sampling variability.",
    "Paired t-test and Wilcoxon signed-rank are reported; with n=5 the smallest two-sided Wilcoxon p is 0.0625, so the t-test (and Holm-adjusted p) is the primary test.",
    "Holm correction is applied within each (Dataset, Family, Metric) group. 'significant' means Holm-adjusted p < ALPHA.",
    "Hyper-parameters are fixed after the first seed when TUNE_ONCE=True.",
]})

TABLES["T00_environment_and_config"] = pd.DataFrame([dict(Key=k, Value=str(v)) for k, v in {**ENV, **{k: v for k, v in CFG.items()},
                                                      "RF_GRID": RF_GRID, "XGB_GRID": XGB_GRID}.items()])

# save
xlsx = os.path.join(OUTPUT_DIR, "chapter4_results.xlsx")
with pd.ExcelWriter(xlsx, engine="openpyxl") as xw:
    for name, t in TABLES.items():
        t.to_csv(os.path.join(TAB_DIR, f"{name}.csv"), index=False)
        t.to_excel(xw, sheet_name=name[:31], index=False)
print("Saved", len(TABLES), "tables ->", xlsx)

for name in ["T03_dataset_summary", "T04_feature_selection", "T08_RQ1_models", "T09_RQ2_feature_selection_effect",
             "T10_RQ3_class_balancing", "T13_RQ4_ensemble_weights", "T17_performance_targets", "T18_cross_dataset_primary"]:
    print("\n", name)
    display(TABLES[name].round(4) if hasattr(TABLES[name], "round") else TABLES[name])

### Figures

In [ ]:
dsets = list(RESULTS.keys())

# Fig 1: class distribution (training set, log-scale)
fig, axes = plt.subplots(1, len(dsets), figsize=(5.6 * len(dsets), 4.6))
for ax, d in zip(np.atleast_1d(axes), dsets):
    s = cd[cd.Dataset == d].sort_values("Train", ascending=False)
    ax.bar(range(len(s)), s.Train, color="#2a6f97")
    ax.set_yscale("log")
    ax.set_xticks(range(len(s)))
    ax.set_xticklabels(s.Class, rotation=75, ha="right", fontsize=8)
    ax.set_title(d)
    ax.set_ylabel("Training records (log scale)")
savefig("fig01_class_distribution.png")

# Fig 2: feature reduction d0 -> d1 -> d2
fsm = fs.groupby("Dataset", sort=False)[["d0", "d1", "d2"]].mean()
fsm.plot(kind="bar", figsize=(6.5, 4), color=["#adb5bd", "#6c9bd2", "#2a6f97"])
plt.ylabel("Number of features"); plt.title("Two-phase feature selection"); plt.xticks(rotation=0)
savefig("fig02_feature_reduction.png")

# Fig 3: Macro-F1 / Weighted-F1 heatmaps (all configurations)
piv = num.assign(Config=lambda x: x.Model + " | " + x.Features + " | " + x.Balancing)
fig, axes = plt.subplots(1, 2, figsize=(13, 0.55 * piv.Config.nunique() + 2))
for ax, m in zip(axes, ["F1_macro", "F1_weighted"]):
    sns.heatmap(piv.pivot_table(index="Config", columns="Dataset", values=m, sort=False), annot=True, fmt=".3f",
                cmap="YlGnBu", vmin=0, vmax=1, ax=ax, cbar=False)
    ax.set_title(m.replace("_", " ")); ax.set_ylabel("")
savefig("fig03_f1_all_configurations.png")

# Fig 4: normalised confusion matrices (primary ensemble)
for d in dsets:
    cfgname = "Ensemble (RF+XGB) | Selected | Original"
    cm = RESULTS[d]["cms"].get(cfgname)
    if cm is None:
        continue
    cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    K = len(RESULTS[d]["classes"])
    plt.figure(figsize=(max(6, 0.55 * K + 3), max(5, 0.5 * K + 2.5)))
    sns.heatmap(cmn, annot=K <= 16, fmt=".2f", cmap="Blues", xticklabels=RESULTS[d]["classes"],
                yticklabels=RESULTS[d]["classes"], vmin=0, vmax=1, annot_kws={"size": 7})
    plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(f"{d}: confusion matrix (row-normalised) — RF+XGBoost ensemble")
    savefig(f"fig04_confusion_{d}.png")

# Fig 5: per-class F1 under balancing conditions
for d in dsets:
    s = pc[pc.Dataset == d]
    s = s[s.Config.str.contains("Selected") & ~s.Config.str.contains("Ensemble")]
    if s.empty:
        continue
    h = s.pivot_table(index="Class", columns="Config", values="F1", aggfunc="mean")
    h.columns = [c.replace(" | Selected", "") for c in h.columns]
    plt.figure(figsize=(8, max(3.5, 0.38 * len(h) + 1.5)))
    sns.heatmap(h, annot=True, fmt=".2f", cmap="YlGnBu", vmin=0, vmax=1, annot_kws={"size": 8})
    plt.title(f"{d}: per-class F1 by model and balancing condition"); plt.ylabel("")
    savefig(f"fig05_perclass_f1_{d}.png")

# Fig 6: ensemble weight search (validation)
wt = ALL("weights")
fig, axes = plt.subplots(1, len(dsets), figsize=(5 * len(dsets), 3.8), sharey=False)
for ax, d in zip(np.atleast_1d(axes), dsets):
    for bal, g in wt[wt.Dataset == d].groupby("Balancing"):
        g = g.groupby("w_RF").Val_F1_macro.mean()
        ax.plot(g.index, g.values, marker="o", label=bal)
    ax.set_title(d); ax.set_xlabel("w_RF (w_XGB = 1 − w_RF)"); ax.set_ylabel("Validation Macro-F1"); ax.legend(fontsize=8)
savefig("fig06_ensemble_weight_search.png")

# Fig 8: Macro-F1 mean ± SD by training condition (ensemble)
ens_ms = res[(res.Model == "Ensemble (RF+XGB)") & (res.Features == "Selected")].groupby(["Dataset", "Balancing"])["F1_macro"].agg(["mean", "std"]).reset_index()
fig, axes = plt.subplots(1, len(dsets), figsize=(4.6 * len(dsets), 4))
for ax, d in zip(np.atleast_1d(axes), dsets):
    s_ = ens_ms[ens_ms.Dataset == d]
    ax.bar(s_.Balancing, s_["mean"], yerr=s_["std"].fillna(0), capsize=4, color="#2a6f97")
    ax.set_ylim(0, 1); ax.set_title(d, fontsize=10); ax.set_ylabel("Macro F1 (mean ± SD)")
    ax.tick_params(axis="x", rotation=25)
savefig("fig08_macroF1_mean_sd.png")

# Fig 7: temporal
for d, r in TEMPORAL.items():
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
    axes[0].plot(r["y_true"], label="Actual", color="black", lw=1.4)
    axes[0].plot(r["preds"]["LSTM"], label="LSTM", lw=1)
    axes[0].plot(r["preds"]["Bi-LSTM"], label="Bi-LSTM", lw=1)
    axes[0].set_title(f"{d}: attack frequency forecast (test period)")
    axes[0].set_xlabel(f"Window index ({CFG['WINDOW']})"); axes[0].set_ylabel("Attacks per window"); axes[0].legend()
    for k, h in r["hist"].items():
        axes[1].plot(h["loss"], label=f"{k} train"); axes[1].plot(h["val_loss"], "--", label=f"{k} val")
    axes[1].set_title("Training / validation loss (MSE, scaled)"); axes[1].set_xlabel("Epoch"); axes[1].legend(fontsize=8)
    savefig(f"fig07_temporal_{d}.png")

## 10) Key findings summary + download everything

In [ ]:
lines = ["# Key numbers for Chapter 4 (auto-generated — verify before writing)\n"]
for d in dsets:
    r = TABLES["T18_cross_dataset_primary"].query("Dataset==@d")
    b = TABLES["T19_best_configuration_per_dataset"].query("Dataset==@d")
    if len(r):
        r = r.iloc[0]
        lines.append(f"- **{d}** ensemble (selected features, original dist.): Macro-F1={r.F1_macro:.4f}, Weighted-F1={r.F1_weighted:.4f}, "
                     f"Accuracy={r.Accuracy:.4f}")
    if len(b):
        b = b.iloc[0]
        lines.append(f"  - best config by Macro-F1: {b.Model} | {b.Features} | {b.Balancing} (Macro-F1={b.F1_macro:.4f})")
    f_ = fs[fs.Dataset == d].iloc[0]
    lines.append(f"  - features: d0={f_.d0} -> d1={f_.d1} -> d2={f_.d2}")
for d, r in TEMPORAL.items():
    t = r["metrics"].groupby("Model")[["MAE", "RMSE"]].mean().round(3)
    lines.append(f"- **Temporal {d}**: " + "; ".join(f"{m}: MAE={v.MAE}, RMSE={v.RMSE}" for m, v in t.iterrows()))
txt = "\n".join(lines)
open(os.path.join(OUTPUT_DIR, "key_findings.md"), "w").write(txt)
print(txt)

In [ ]:
from google.colab import files
zip_path = shutil.make_archive(os.path.join("/content", "Wa_la_Yahimk_chapter4_outputs"), "zip", OUTPUT_DIR)
print("Zipped:", zip_path)
files.download(zip_path)